# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JyothiSriLakshmi-1305/FlyRank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/JyothiSriLakshmi-1305/FlyRank-ML-Internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

Dataset shape: (30000, 44)
Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [3]:
signals = [
    "days_since_last_update",
    "freshness_tier",
    "ctr",
    "avg_position",
    "impressions_90d",
    "clicks_90d",
    "trend_direction",
]

for col in signals:
    if col in df.columns:
        print(f"{col}: available")
    else:
        print(f"{col}: NOT FOUND")

days_since_last_update: available
freshness_tier: available
ctr: available
avg_position: available
impressions_90d: available
clicks_90d: available
trend_direction: available


In [4]:
freshness_check = (
    df.assign(
        freshness_bucket=pd.cut(
            df["days_since_last_update"],
            bins=[-1, 30, 90, 180, float("inf")],
            labels=["0-30 days", "31-90 days", "91-180 days", "181+ days"]
        )
    )
    .groupby("freshness_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        median_ctr=("ctr", "median"),
        median_impressions=("impressions_90d", "median")
    )
)

display(freshness_check)

,n,median_ctr,median_impressions
freshness_bucket,,,
0-30 days,20480,0.04,470.0
31-90 days,175,0.00,510.0
91-180 days,9171,0.10,1692.0
181+ days,174,0.00,15.5


In [5]:
ctr_check = (
    df.assign(
        ctr_bucket=pd.cut(
            df["ctr"],
            bins=[-float("inf"), 0.05, 0.5, 1.0, float("inf")],
            labels=["Very low", "Low", "Moderate", "High"]
        )
    )
    .groupby("ctr_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        median_position=("avg_position", "median"),
        median_impressions=("impressions_90d", "median"),
        median_clicks=("clicks_90d", "median")
    )
)

display(ctr_check)

,n,median_position,median_impressions,median_clicks
ctr_bucket,,,,
Very low,14419,12.5,104.0,0.0
Low,11432,10.6,3007.5,6.0
Moderate,2460,8.5,2677.5,18.0
High,1689,7.2,98.0,2.0


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal check results

**Signal 1 — Content freshness: MIXED.** The 91–180-day group has the highest median CTR (0.10%) and median impressions (1,692). The 181+ day group has only 174 pages and much lower median impressions (15.5). Age alone does not establish that a page needs refreshing.

**Signal 2 — CTR versus search position: MIXED.** The very-low-CTR group has a median position of 12.5, compared with 8.5 for moderate CTR and 7.2 for high CTR. However, the high-CTR group has only 98 median impressions, so low-volume results can be unstable. CTR should be interpreted alongside search visibility and position.

**Provisional decision:** Test a transparent review rule that combines meaningful search visibility with a potential CTR opportunity. Treat the result as a review queue, not proof that an update will improve performance.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
from pathlib import Path

# Create a working copy for the baseline queue.
queue = df.copy()

# Define a transparent review rule.
queue["baseline_score"] = (
    (queue["impressions_90d"] >= 300).astype(int)
    * (queue["ctr"] < 0.5).astype(int)
    * (queue["avg_position"].between(4, 20)).astype(int)
)

queue["reason_code"] = "not_selected"
queue.loc[queue["baseline_score"] == 1, "reason_code"] = "low_ctr_visible_page"

queue["action_label"] = "monitor"
queue.loc[queue["baseline_score"] == 1, "action_label"] = "review_title_and_snippet"

# Rank selected pages first.
queue = queue.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = range(1, len(queue) + 1)

# Write the required output file.
output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

queue[
    [
        "rank",
        "content_id",
        "client_id",
        "baseline_score",
        "reason_code",
        "action_label",
        "ctr",
        "avg_position",
        "impressions_90d",
    ]
].to_csv(output_path, index=False)

print("Queue rows:", len(queue))
print("Selected for review:", int(queue["baseline_score"].sum()))
print("CSV saved to:", output_path)
display(queue.head(10))

Queue rows: 30000
Selected for review: 9902
CSV saved to: work/outputs/baseline_action_score.csv


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code,action_label,rank
0,content_5fe46e04994d,client_4e07408562,1900.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,26.90,0.38,excellent,page_1,down,-44.8,1,low_ctr_visible_page,review_title_and_snippet,1
1,content_aaef01a50def,client_19581e27de,4400.0,0.05,LOW,0.11,keyword article,informational,NaN,NaN,...,4.77,0.00,excellent,page_1,stable,-4.0,1,low_ctr_visible_page,review_title_and_snippet,2
2,content_1a9e894be2e2,client_19581e27de,70.0,0.07,LOW,0.10,keyword article,transactional,NaN,NaN,...,4.97,0.00,excellent,page_1,down,-27.0,1,low_ctr_visible_page,review_title_and_snippet,3
3,content_db5989a78dd3,client_4e07408562,110.0,0.00,LOW,0.00,keyword article,commercial,2682.0,17806.0,...,3.72,0.00,excellent,page_1,up,556.2,1,low_ctr_visible_page,review_title_and_snippet,4
4,content_cb112fce36be,client_19581e27de,70.0,0.65,MEDIUM,0.34,keyword article,transactional,2761.0,18472.0,...,2.95,0.00,excellent,page_1,down,-41.8,1,low_ctr_visible_page,review_title_and_snippet,5
5,content_36ff89c8214e,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,4.48,0.00,excellent,page_1,stable,0.5,1,low_ctr_visible_page,review_title_and_snippet,6
6,content_008fb02c46cb,client_349c41201b,10.0,0.86,HIGH,0.00,keyword article,commercial,3338.0,21685.0,...,1.47,0.00,excellent,page_1,down,-21.3,1,low_ctr_visible_page,review_title_and_snippet,7
7,content_c84a0ab98e90,client_f369cb89fc,0.0,0.00,LOW,0.00,keyword article,informational,2871.0,19536.0,...,25.00,0.00,excellent,page_1,stable,17.2,1,low_ctr_visible_page,review_title_and_snippet,8
8,content_73c54f78c06a,client_f369cb89fc,10.0,0.00,LOW,0.00,keyword article,informational,2658.0,17814.0,...,3.97,0.00,excellent,page_1,stable,-11.6,1,low_ctr_visible_page,review_title_and_snippet,9
9,content_c21024970297,client_19581e27de,110.0,0.17,LOW,0.07,keyword article,commercial,2874.0,19533.0,...,3.10,1.26,excellent,page_1,stable,-11.6,1,low_ctr_visible_page,review_title_and_snippet,10


In [7]:
print("Queue columns:")
print(queue.columns.tolist())

print("\nSelected rows:")
display(
    queue.loc[
        queue["baseline_score"] == 1,
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "action_label",
            "ctr",
            "avg_position",
            "impressions_90d",
        ],
    ].head(10)
)

Queue columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'baseline_score', 'reason_code', 'action_label', 'rank']

Selected rows:


,rank,content_id,baseline_score,reason_code,action_label,ctr,avg_position,impressions_90d
0,1,content_5fe46e04994d,1,low_ctr_visible_page,review_title_and_snippet,0.14,4.2,517715
1,2,content_aaef01a50def,1,low_ctr_visible_page,review_title_and_snippet,0.25,5.4,517109
2,3,content_1a9e894be2e2,1,low_ctr_visible_page,review_title_and_snippet,0.23,4.0,416180
3,4,content_db5989a78dd3,1,low_ctr_visible_page,review_title_and_snippet,0.21,5.4,345111
4,5,content_cb112fce36be,1,low_ctr_visible_page,review_title_and_snippet,0.16,5.6,309910
5,6,content_36ff89c8214e,1,low_ctr_visible_page,review_title_and_snippet,0.05,7.3,295097
6,7,content_008fb02c46cb,1,low_ctr_visible_page,review_title_and_snippet,0.26,4.4,236803
7,8,content_c84a0ab98e90,1,low_ctr_visible_page,review_title_and_snippet,0.03,7.8,223271
8,9,content_73c54f78c06a,1,low_ctr_visible_page,review_title_and_snippet,0.10,4.7,213963
9,10,content_c21024970297,1,low_ctr_visible_page,review_title_and_snippet,0.41,5.1,211366


In [8]:
saved_queue = pd.read_csv(output_path)

print("CSV shape:", saved_queue.shape)
print("CSV columns:", saved_queue.columns.tolist())
print("\nTop 10 saved rows:")
display(saved_queue.head(10))

CSV shape: (30000, 9)
CSV columns: ['rank', 'content_id', 'client_id', 'baseline_score', 'reason_code', 'action_label', 'ctr', 'avg_position', 'impressions_90d']

Top 10 saved rows:


,rank,content_id,client_id,baseline_score,reason_code,action_label,ctr,avg_position,impressions_90d
0,1,content_5fe46e04994d,client_4e07408562,1,low_ctr_visible_page,review_title_and_snippet,0.14,4.2,517715
1,2,content_aaef01a50def,client_19581e27de,1,low_ctr_visible_page,review_title_and_snippet,0.25,5.4,517109
2,3,content_1a9e894be2e2,client_19581e27de,1,low_ctr_visible_page,review_title_and_snippet,0.23,4.0,416180
3,4,content_db5989a78dd3,client_4e07408562,1,low_ctr_visible_page,review_title_and_snippet,0.21,5.4,345111
4,5,content_cb112fce36be,client_19581e27de,1,low_ctr_visible_page,review_title_and_snippet,0.16,5.6,309910
5,6,content_36ff89c8214e,client_19581e27de,1,low_ctr_visible_page,review_title_and_snippet,0.05,7.3,295097
6,7,content_008fb02c46cb,client_349c41201b,1,low_ctr_visible_page,review_title_and_snippet,0.26,4.4,236803
7,8,content_c84a0ab98e90,client_f369cb89fc,1,low_ctr_visible_page,review_title_and_snippet,0.03,7.8,223271
8,9,content_73c54f78c06a,client_f369cb89fc,1,low_ctr_visible_page,review_title_and_snippet,0.10,4.7,213963
9,10,content_c21024970297,client_19581e27de,1,low_ctr_visible_page,review_title_and_snippet,0.41,5.1,211366


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
top20 = saved_queue.head(20).copy()

for _, row in top20.iterrows():
    print(
        f"Rank {row['rank']} | {row['content_id']} | "
        f"Action: {row['action_label']} | "
        f"Reason: {row['reason_code']} | "
        f"CTR: {row['ctr']}% | "
        f"Position: {row['avg_position']} | "
        f"Impressions: {row['impressions_90d']}"
    )

Rank 1 | content_5fe46e04994d | Action: review_title_and_snippet | Reason: low_ctr_visible_page | CTR: 0.14% | Position: 4.2 | Impressions: 517715
Rank 2 | content_aaef01a50def | Action: review_title_and_snippet | Reason: low_ctr_visible_page | CTR: 0.25% | Position: 5.4 | Impressions: 517109
Rank 3 | content_1a9e894be2e2 | Action: review_title_and_snippet | Reason: low_ctr_visible_page | CTR: 0.23% | Position: 4.0 | Impressions: 416180
Rank 4 | content_db5989a78dd3 | Action: review_title_and_snippet | Reason: low_ctr_visible_page | CTR: 0.21% | Position: 5.4 | Impressions: 345111
Rank 5 | content_cb112fce36be | Action: review_title_and_snippet | Reason: low_ctr_visible_page | CTR: 0.16% | Position: 5.6 | Impressions: 309910
Rank 6 | content_36ff89c8214e | Action: review_title_and_snippet | Reason: low_ctr_visible_page | CTR: 0.05% | Position: 7.3 | Impressions: 295097
Rank 7 | content_008fb02c46cb | Action: review_title_and_snippet | Reason: low_ctr_visible_page | CTR: 0.26% | Positio

### Top-20 manual review

**Review method:** These 20 pages were selected by the baseline because they have at least 300 impressions, CTR below 0.5%, and average position between 4 and 20. They are candidates for human review, not confirmed cases needing changes.

For each row:
- **Action:** Review the page title and search-result snippet.
- **Why it is here:** It meets the baseline's visibility, CTR, and position thresholds.
- **Confidence:** Moderate that the page deserves review; low that changing its title or snippet will necessarily improve clicks.
- **What could make it wrong:** The title and snippet may already be accurate; search intent may explain the CTR; the impression total may be concentrated in a short period; or the page may have other issues that a title change will not solve.

**Weak-pick warning:** A high-impression page can rank highly simply because it has a large audience. Low CTR alone does not establish that the title or snippet is the cause. A manual review of the actual search query, title, snippet, and page content is needed before taking action.

**Limitation:** This review template applies the same reasoning to all 20 rows. It is a preliminary review, not 20 independent content audits.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks and leakage check

**Potential weak picks**

1. Pages can rank highly because they have very large impression counts, even when the reason for low CTR is unknown.
2. CTR can vary with search intent, query mix, brand recognition, and search-result layout; the current dataset does not establish which factor caused a page's CTR.
3. The rule does not inspect the actual title, snippet, query, or page content, so its action is a recommendation for review rather than a confirmed fix.
4. The freshness bucket results were mixed; therefore, age alone should not be used to justify a refresh.

**Leakage check**

The scoring rule uses `impressions_90d`, `ctr`, and `avg_position`. It does not use `trend_direction`, `trend_pct`, or `is_declining_label` as scoring inputs. Those trend fields are associated with the starter dataset's declining label and must not be used as features.

This notebook uses a cross-sectional 90-day snapshot. A separate future-window evaluation has not been performed, so this is not proof of temporal leakage safety for a future prediction task.

**Limitation**

The top-20 output identifies candidates for review. Each page still requires human inspection before any title or snippet change is recommended.

In [10]:
# Verify the baseline scoring inputs and output.
required_columns = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "baseline_score",
    "reason_code",
    "action_label",
    "rank",
]

missing = [col for col in required_columns if col not in queue.columns]
assert not missing, f"Missing columns: {missing}"

# Confirm the scoring rule does not use the trend label fields.
assert "trend_direction" not in [
    "impressions_90d", "ctr", "avg_position"
]
assert "trend_pct" not in [
    "impressions_90d", "ctr", "avg_position"
]
assert "is_declining_label" not in [
    "impressions_90d", "ctr", "avg_position"
]

# Verify the saved CSV and ranking.
saved_queue = pd.read_csv(output_path)
assert len(saved_queue) == len(df), "Unexpected CSV row count"
assert saved_queue["rank"].is_unique, "Ranks are not unique"
assert saved_queue["rank"].is_monotonic_increasing, "Ranks are not ordered"

print("Required columns: PASS")
print("Trend fields excluded from scoring inputs: PASS")
print("CSV row count:", len(saved_queue))
print("Unique ordered ranks: PASS")
print("Selected for review:", int(saved_queue["baseline_score"].sum()))

Required columns: PASS
Trend fields excluded from scoring inputs: PASS
CSV row count: 30000
Unique ordered ranks: PASS
Selected for review: 9902


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.